# Lab 1 – Parsing PDF bảo hiểm tiếng Việt

**Mục tiêu:** parse cùng tài liệu bằng PyMuPDF (text thô) và Docling (Markdown), so sánh 3 thứ hay làm hỏng RAG bảo hiểm:
1. **Bảng** quyền lợi / biểu phí (giữ được dòng–cột không?)
2. **Thứ tự đọc** trang nhiều cột
3. **Dấu tiếng Việt** (NFC, ký tự lỗi)

Ngoài ra: trang đôi (An Gia, Tâm An: 1 trang PDF = 2 trang in), header/footer lặp, tài liệu gần trùng.

Đầu ra: `data/parsed/{doc_id}.{parser}.json|md` cho cả corpus (Lab 2+ dùng bản `docling`), `reports/lab1_results.json`.

In [1]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_colwidth", 120)

from ragbv.config import load_manifest, REPORTS
from ragbv import parsing as P, eval_parsing as E

docs = load_manifest()
pd.DataFrame([{"doc_id": d.doc_id, "layout": d.page_layout, "type": d.doc_type, "file": d.file} for d in docs.values()])

,doc_id,layout,type,file
0,bv-angia-2016,spread,quy_tac,raw/BaoVietInsurance_AnGia_QuyTac_2016.pdf
1,bv-taman-2019,spread,quy_tac,raw/BaoVietInsurance_TamAn_QuyTac_2019.pdf
2,bvl-avsk-2022,single,quy_tac,raw/BaoVietLife_AnVuiSongKhoe_QuyTacDieuKhoan_2022.pdf
3,bvl-avsk-tomtat-2022,single,tom_tat,raw/BaoVietLife_AnVuiSongKhoe_TomTat_2022.pdf
4,manulife-cssknc-2026,single,quy_tac,raw/Manulife_ChamSocSucKhoeNangCao_QuyTacDieuKhoan_2026.pdf
5,daiichi-tktt2015-2025,single,quy_tac,raw/DaiichiLife_TuKyTaiTucHangThang2015_QuyTacDieuKhoan_2025.pdf


## 1. Trang đôi → trang logic
Trang ngang được chia nửa trái/phải; số trang in đánh tuần tự để trích dẫn đúng số in trên giấy. Trang khai báo `keep_spread_pages` (bảng trải ngang) được giữ nguyên.

In [2]:
for doc_id in ["bv-angia-2016", "bv-taman-2019"]:
    lps = P.doc_logical_pages(docs[doc_id])
    print(doc_id, "PDF pages →", len(lps), "logical pages")
    print("  ", [(lp.pdf_page, lp.half, lp.page_label) for lp in lps if lp.pdf_page in (1, 2, 13, 14)])

bv-angia-2016 PDF pages → 27 logical pages
   [(1, 'full', '1'), (2, 'left', '2'), (2, 'right', '3'), (13, 'spread', '24-25'), (14, 'left', '26'), (14, 'right', '27')]
bv-taman-2019 PDF pages → 28 logical pages
   [(1, 'full', '1'), (2, 'left', '2'), (2, 'right', '3'), (13, 'left', '24'), (13, 'right', '25'), (14, 'left', '26'), (14, 'right', '27')]


## 2. Parse toàn corpus bằng 3 cách
`pymupdf` = thứ tự content stream, `pymupdf_sort` = sắp theo toạ độ, `docling` = layout model + TableFormer (tắt OCR vì PDF có text layer). Sau đó chuẩn hoá NFC + bỏ header/footer lặp.

In [3]:
results, parsed, raw, removed = [], {}, {}, {}
for doc_id, meta in docs.items():
    for parser in ["pymupdf", "pymupdf_sort", "docling"]:
        raw_pages, secs = P.run_parser(meta, parser)
        # Docling: dùng thêm header/footer phát hiện từ lớp text PyMuPDF (parser chạy trước trong vòng lặp)
        extra = set(removed[(doc_id, "pymupdf")]) if parser == "docling" else None
        pages, rm = P.clean_pages(raw_pages, extra)
        raw[(doc_id, parser)] = raw_pages
        P.save_pages(doc_id, parser, pages)
        parsed[(doc_id, parser)], removed[(doc_id, parser)] = pages, rm
        results.append({"doc_id": doc_id, "parser": parser, "pages": len(pages), "seconds": round(secs, 1),
                        "removed_header_footer_lines": len(rm)})
timing = pd.DataFrame(results)
timing.pivot(index="doc_id", columns="parser", values="seconds")

C:\Users\Legion\Desktop\Job\Paraline\Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:  34%|███▍      | 262/770 [00:00<00:00, 2566.11it/s]

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 3930.47it/s]

parser,docling,pymupdf,pymupdf_sort
doc_id,,,
bv-angia-2016,26.5,0.2,0.8
bv-taman-2019,5.7,0.1,0.5
bvl-avsk-2022,14.7,0.1,0.5
bvl-avsk-tomtat-2022,0.4,0.0,0.0
daiichi-tktt2015-2025,3.4,0.1,0.5
manulife-cssknc-2026,13.0,0.2,0.7


## 3. Bảng: biểu phí An Gia, biểu phí Tâm An, bảng quyền lợi An Gia
- `value_recall`: số trong bảng có xuất hiện trên trang không (text có ra không)
- `row_integrity`: số có nằm **đúng dòng, đúng thứ tự cột** không (cấu trúc có giữ được không) — đây mới là thứ quyết định LLM đọc đúng phí của chương trình nào

In [4]:
rows = []
for test, t in E.TABLE_TESTS.items():
    for parser in ["pymupdf", "pymupdf_sort", "docling"]:
        rows.append({"test": test, "parser": parser, **E.table_report(parsed[(t["doc_id"], parser)], test)})
    # PyMuPDF find_tables(): phát hiện bảng theo đường kẻ
    meta = docs[t["doc_id"]]
    tabs = P.pymupdf_tables(meta, t["page_label"])
    page_text = next(p.text for p in parsed[(t["doc_id"], "pymupdf")] if p.page_label == t["page_label"])
    rows.append({"test": test, "parser": "pymupdf_find_tables",
                 **E.table_report_rows([r for tab in tabs for r in tab], test, page_text)})
    # Bỏ qua lỗi ký tự trong ô (mất khoảng trắng, dấu chấm tách khỏi số) để chỉ đo cấu trúc lưới
    rows.append({"test": test, "parser": "pymupdf_find_tables_lenient",
                 **E.table_report_rows([r for tab in tabs for r in tab], test, page_text, lenient=True)})
tables = pd.DataFrame(rows)
tables.pivot(index="parser", columns="test", values="row_integrity")

test,angia_bieu_phi,angia_quyen_loi,taman_bieu_phi
parser,,,
docling,1.0,1.0,1.0
pymupdf,0.0,0.0,0.0
pymupdf_find_tables,0.0,0.0,0.0
pymupdf_find_tables_lenient,1.0,1.0,1.0
pymupdf_sort,1.0,1.0,1.0


In [5]:
tables.pivot(index="parser", columns="test", values="value_recall")

test,angia_bieu_phi,angia_quyen_loi,taman_bieu_phi
parser,,,
docling,1.0,1.0,1.0
pymupdf,1.0,1.0,1.0
pymupdf_find_tables,1.0,1.0,1.0
pymupdf_find_tables_lenient,1.0,1.0,1.0
pymupdf_sort,1.0,1.0,1.0


### 3b. Tách trang đôi "ngây thơ" cắt đôi bảng quyền lợi
So sánh Docling khi tách mọi trang đôi (`split=all`) với khi giữ nguyên trang 13 (`split=manifest`).

In [6]:
naive, _ = P.run_parser(docs["bv-angia-2016"], "docling", split="all")
naive, _ = P.clean_pages(naive)
# Ở chế độ 'all', bảng nằm trên 2 trang logic 24 và 25 -> gộp lại để chấm công bằng
merged = [P.Page(0, 13, "spread", "24-25", "\n".join(p.text for p in naive if p.page_label in ("24", "25")))]
split_cmp = pd.DataFrame([
    {"mode": "split=all (naive)", **E.table_report(merged, "angia_quyen_loi")},
    {"mode": "split=manifest (giữ trang 13)", **E.table_report(parsed[("bv-angia-2016", "docling")], "angia_quyen_loi")},
])
split_cmp

,mode,cells,value_recall,row_integrity
0,split=all (naive),22,1.0,0.227
1,split=manifest (giữ trang 13),22,1.0,1.000


In [7]:
print(next(p.text for p in naive if p.page_label == "24")[:1500])

## BẢNG QUYỀN LỢI BẢO

(Ban hành theo Quyết định triển khai số 4542 ngày 13 tháng 10

| CHƯƠNG TRÌNH BẢO HIỂM | ĐỒNG | |
|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------|---------------------------------------------------------------------|
| A. ĐIỀU KHOẢN BẢO HIỂM CHÍNH - Điều trị nội trú do ốm bệnh, tai nạn | A. ĐIỀU KHOẢN BẢO HIỂM CHÍNH - Điều trị nội trú do ốm bệnh, tai nạn | A. ĐIỀU KHOẢN BẢO HIỂM CHÍNH - Điều trị nội trú do ốm bệnh, tai nạn |
| Tổng hạn mức quyền lợi bảo hiểm chính/người/năm | 93.800.000/ng/năm | 137.600 |
| Phạm vi lãnh thổ | Việt Nam | Vi |
| 1. Chi phí nằm viện Chi phí phòng, giường bệnh Chi phí phòng chăm sóc đặc biệt Các chi phí bệnh viện tổng hợp | 2.000.000/ngày Lên đến 40.000.000/năm, không quá 6

### 3c. Xem trực tiếp output biểu phí An Gia (trang in 26)

In [8]:
for parser in ["pymupdf", "docling"]:
    print(f"===== {parser} =====")
    print(next(p.text for p in parsed[("bv-angia-2016", parser)] if p.page_label == "26")[:1200])

===== pymupdf =====
26
Đơn vị: VND/người/năm
CHƯƠNG TRÌNH
BẢO HIỂM
ĐỒNG
BẠC
VÀNG
BẠCH KIM
KIM CƯƠNG
A. ĐIỀU KHOẢN BẢO HIỂM CHÍNH – Điều trị nội trú do ốm bệnh, tai nạn
•
Từ 1 đến 3 tuổi
2.585.000
3.707.000
5.616.000
7.582.000
9.267.000
•
Từ 4 đến 6 tuổi
1.450.000
2.081.000
3.153.000
4.257.000
5.203.000
•
Từ 7 đến 9 tuổi
1.320.000
1.900.000
2.867.000
3.870.000
4.730.000
•
Từ 10 đến 18 tuổi
1.255.800
1.801.800
2.730.000
3.685.500
4.504.500
•
Từ 19 đến 30 tuổi
1.196.000
1.716.000
2.600.000
3.510.000
4.290.000
•
Từ 31 đến 40 tuổi
1.315.600
1.887.600
2.860.000
3.861.000
4.719.000
•
Từ 41 đến 50 tuổi
1.375.400
1.973.400
2.990.000
4.036.500
4.933.500
•
Từ 51 đến 60 tuổi
1.435.200
2.059.200
3.120.000
4.212.000
5.148.000
•
Từ 61 đến 65 tuổi
1.554.800
2.230.800
3.380.000
4.563.000
5.577.000
B. ĐIỀU KHOẢN BẢO HIỂM BỔ SUNG
1. Điều trị ngoại trú do
ốm bệnh, tai nạn
ĐỒNG
BẠC
VÀNG
BẠCH KIM
KIM CƯƠNG
•
Từ 1 đến 3 tuổi
3.370.000
3.370.000
3.370.000
5.516.000
8.424.000
•
Từ 4 đến 6 tuổi
1.900.000
1.900.

## 4. Thứ tự đọc: định nghĩa An Gia (trang in 4–5, 2 cột/trang)
Đúng thứ tự là 1 → 21. `in_order` = độ dài dãy con tăng dài nhất / 21.

In [9]:
order = pd.DataFrame([{"parser": parser, **E.reading_order(parsed[("bv-angia-2016", parser)])}
                      for parser in ["pymupdf", "pymupdf_sort", "docling"]])
order

,parser,found,in_order,order
0,pymupdf,21/21,1.000,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]"
1,pymupdf_sort,21/21,0.619,"[6, 1, 7, 2, 8, 3, 9, 4, 10, 5, 11, 12, 19, 13, 20, 14, 15, 21, 16, 17, 18]"
2,docling,21/21,1.000,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]"


## 5. Dấu tiếng Việt & chất lượng text

In [10]:
quality = pd.DataFrame([{"doc_id": d, "parser": p, **E.text_quality(pages, raw[(d, p)])} for (d, p), pages in parsed.items()])
quality

,doc_id,parser,chars,replacement_chars,private_use_chars,non_nfc_marks,control_chars_before_clean,'bảo hiểm','người được bảo hiểm','quyền lợi',empty_pages
0,bv-angia-2016,pymupdf,66477,0,0,0,43,450,72,52,1
1,bv-angia-2016,pymupdf_sort,67076,0,0,0,0,451,72,52,1
2,bv-angia-2016,docling,73121,0,0,0,44,505,92,57,1
3,bv-taman-2019,pymupdf,53056,0,0,0,0,461,109,43,3
4,bv-taman-2019,pymupdf_sort,54062,0,0,0,0,461,109,43,3
5,bv-taman-2019,docling,54967,0,0,0,0,470,118,45,2
6,bvl-avsk-2022,pymupdf,49381,0,0,0,0,283,55,120,1
7,bvl-avsk-2022,pymupdf_sort,50230,0,0,0,0,283,55,120,1
8,bvl-avsk-2022,docling,58825,0,0,0,0,310,72,160,1
9,bvl-avsk-tomtat-2022,pymupdf,3033,0,0,0,0,32,2,12,0


In [11]:
# Thử nghiệm NFC: cùng chữ 'bảo hiểm' ở dạng dựng sẵn (NFC) và tổ hợp (NFD) -> so khớp chuỗi thất bại
import unicodedata
nfc, nfd = "bảo hiểm", unicodedata.normalize("NFD", "bảo hiểm")
print(len(nfc), len(nfd), nfc == nfd, nfc in ("hợp đồng " + nfd), P.normalize(nfd) == nfc)

8 11 False False True


## 6. Header/footer đã bỏ

In [12]:
for doc_id in docs:
    print(doc_id, removed[(doc_id, "docling")][:8])

bv-angia-2016 ['bảo hiểm “bảo việt an gia”', 'www.baoviet.com.vn/baohiem']
bv-taman-2019 ['bảo việt tâm an | an tâm sống vui sống khỏe', 'w w w. b a o v i e t o n l i n e . c o m . v n']
bvl-avsk-2022 ['<!-- image -->', 'bao viet life', 'bảo việt nhân thọ', 'quy tắc, điều khoản sản phẩm bổ trợ bảo hiểm chăm sóc sức khỏe']
bvl-avsk-tomtat-2022 []
manulife-cssknc-2026 ['<!-- image -->', 'sản phẩm bảo hiểm chăm sóc sức khỏe nâng cao – tái tục hàng năm']
daiichi-tktt2015-2025 ['#/#', '<!-- image -->', 'quy tắc và điều khoản bảo hiểm nhân thọ tử kỳ tái tục hàng tháng (#)']


## 7. Tài liệu gần trùng: tóm tắt An Vui Sống Khỏe vs trang 1–2 của bản QTĐK
Jaccard trên shingle 5 từ. Cao → khi index cả hai sẽ có chunk trùng chiếm chỗ trong top-k (xử lý ở Lab 2).

In [13]:
import re
def shingles(t, n=5):
    w = re.findall(r"\w+", t.lower()); return {" ".join(w[i:i+n]) for i in range(len(w)-n+1)}
a = shingles("\n".join(p.text for p in parsed[("bvl-avsk-tomtat-2022", "docling")]))
b = shingles("\n".join(p.text for p in parsed[("bvl-avsk-2022", "docling")] if p.pdf_page <= 2))
c = shingles("\n".join(p.text for p in parsed[("bvl-avsk-2022", "docling")] if p.pdf_page > 2))
print("tóm tắt vs QTĐK tr.1-2:", round(len(a & b) / len(a | b), 3), "| tóm tắt vs phần còn lại:", round(len(a & c) / len(a | c), 3))

tóm tắt vs QTĐK tr.1-2: 0.707 | tóm tắt vs phần còn lại: 0.019


## 8. Lưu kết quả

In [14]:
REPORTS.mkdir(exist_ok=True)
out = {"timing": timing.to_dict("records"), "tables": tables.to_dict("records"),
       "split_compare": split_cmp.to_dict("records"), "reading_order": order.to_dict("records"),
       "quality": quality.to_dict("records")}
(REPORTS / "lab1_results.json").write_text(json.dumps(out, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("saved")

saved
